# SpectraRestore — Google Colab

**KLA PS01 · SEMICON India Hackathon 2026**  
Joint denoise + 2× super-resolution (NAFNet-SR2×)

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (or A100/L4 if available)
2. Put the KLA dataset on Drive under `MyDrive/SpectraRestore/data/` with `train/` + `val/` pairs
3. Upload `dist/SpectraRestore.zip` to `MyDrive/SpectraRestore/` (recommended), or let the notebook clone the public repository
4. **Run All** — source code, dependencies, data, checkpoints, and outputs are handled below

## 0 · Check GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Get the project code

Choose either source; no code editing is required:
1. **Recommended:** upload the supplied `SpectraRestore.zip` to `MyDrive/SpectraRestore/`. This works even if GitHub is unavailable.
2. Otherwise, the notebook clones the public repository.

A project already unpacked in the Colab session is reused.

In [ ]:
from pathlib import Path
import zipfile

REPO_URL = 'https://github.com/Charan-suresh/SpectraRestore.git'
PROJECT = Path('/content/SpectraRestore')
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/SpectraRestore/SpectraRestore.zip')

if (PROJECT / 'src' / 'model.py').is_file():
    print(f'Project already present at {PROJECT} — reusing.')
elif DRIVE_ARCHIVE.is_file():
    print(f'Extracting supplied archive: {DRIVE_ARCHIVE}')
    with zipfile.ZipFile(DRIVE_ARCHIVE) as archive:
        archive.extractall(PROJECT)
else:
    print(f'Archive not found; cloning from {REPO_URL} ...')
    !git clone --depth 1 {REPO_URL} {PROJECT}

assert (PROJECT / 'src' / 'model.py').is_file(), (
    f'Project setup failed — src/model.py not found in {PROJECT}\n'
    'Upload SpectraRestore.zip to MyDrive/SpectraRestore/ or check repository access.'
)

%cd {PROJECT}
!ls -la
print(f'\nPROJECT = {PROJECT}')

## 3 · Install dependencies

In [ ]:
# Install the exact project dependencies. Colab keeps its CUDA-enabled PyTorch when compatible.
%pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.model import build_model
m = build_model('default')
print(f"Model OK — {m.num_params()/1e6:.2f}M params")

## 4 · Dataset setup

### Recommended layout on Drive
```
MyDrive/SpectraRestore/data/
  train/
    degraded/   *.png / *.tif / *.npy
    gt/         same filenames
  val/
    degraded/
    gt/
```

Official KLA Drive folder:  
https://drive.google.com/drive/folders/1VKiFW-kDk9-q5XRPu3nrl08OM94EwzV6

Download it into `MyDrive/SpectraRestore/data/`. If it is not available, the next cell creates a small synthetic dataset so the complete notebook can still be run and verified.

In [ ]:
from pathlib import Path
import os
import numpy as np
from PIL import Image

# Point this at wherever you put the KLA data on Drive.
DRIVE_DATA = Path('/content/drive/MyDrive/SpectraRestore/data')
LOCAL_DATA = Path('data')  # relative to PROJECT

if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
    if LOCAL_DATA.is_symlink():
        LOCAL_DATA.unlink()
    else:
        # local data dir exists; keep it as-is
        pass

USING_DEMO_DATA = (LOCAL_DATA / '.spectrarestore_demo').is_file()
if not LOCAL_DATA.exists() and DRIVE_DATA.exists():
    # Symlink so training reads from Drive without copying GBs.
    os.symlink(DRIVE_DATA, LOCAL_DATA)
elif not LOCAL_DATA.exists():
    # Keep Run All useful on a clean Colab session. Replace this with the KLA data for real training.
    USING_DEMO_DATA = True
    rng = np.random.default_rng(42)
    for split, count in (('train', 8), ('val', 2)):
        deg_dir = LOCAL_DATA / split / 'degraded'
        gt_dir = LOCAL_DATA / split / 'gt'
        deg_dir.mkdir(parents=True, exist_ok=True)
        gt_dir.mkdir(parents=True, exist_ok=True)
        y, x = np.mgrid[:128, :128] / 127.0
        for index in range(count):
            phase = rng.uniform(0, 2 * np.pi)
            gt = np.clip(0.5 + 0.22 * np.sin(12 * np.pi * x + phase) * np.cos(10 * np.pi * y - phase) + 0.1 * rng.normal(size=(128, 128)), 0, 1)
            degraded = gt.reshape(64, 2, 64, 2).mean(axis=(1, 3))
            degraded = np.clip(degraded + 0.06 * rng.normal(size=(64, 64)), 0, 1)
            Image.fromarray((gt * 255).astype(np.uint8)).save(gt_dir / f'sample_{index:03d}.png')
            Image.fromarray((degraded * 255).astype(np.uint8)).save(deg_dir / f'sample_{index:03d}.png')
    (LOCAL_DATA / '.spectrarestore_demo').touch()
    print('KLA data not found; created a synthetic demo dataset for this run.')

print('data ->', LOCAL_DATA.resolve())
!find -L data -type f | head -20
print('--- file counts ---')
!find -L data -type f | sed 's|/[^/]*$||' | sort | uniq -c | sort -rn | head

## 5 · Quick smoke test (30 seconds)

In [ ]:
!python scripts/smoke_test.py

## 6 · Train

Tips for Colab free/T4:
- Start with `--preset default --batch_size 4` (or `8` if VRAM allows)
- Use fewer iters for a first pass (`50000`), then resume for full `200000`
- Weights are also copied to Drive so a disconnect doesn't wipe them

In [ ]:
from pathlib import Path

# The synthetic fallback is intentionally short; use the production values with the KLA dataset.
PRESET = 'tiny' if USING_DEMO_DATA else 'default'
BATCH = 2 if USING_DEMO_DATA else 4
ITERS = 10 if USING_DEMO_DATA else 50000
GT_CROP = 128 if USING_DEMO_DATA else 256
VAL_EVERY = 5 if USING_DEMO_DATA else 1000
SAVE_EVERY = 10 if USING_DEMO_DATA else 2000
LOG_EVERY = 5 if USING_DEMO_DATA else 50
NO_LPIPS = ' --no_lpips' if USING_DEMO_DATA else ''

WEIGHTS_LOCAL = Path('weights')
WEIGHTS_DRIVE = Path('/content/drive/MyDrive/SpectraRestore/weights')
WEIGHTS_LOCAL.mkdir(exist_ok=True)
WEIGHTS_DRIVE.mkdir(parents=True, exist_ok=True)

# Resume from the best available checkpoint on Drive
resume = ''

# Prefer full checkpoints (they include optimizer state for proper resume)
ckpts = sorted(WEIGHTS_DRIVE.glob('ckpt_*.pt'))
if ckpts:
    resume = f' --resume {ckpts[-1]}'
    print('Resuming from', ckpts[-1])
else:
    # Fall back to best.pt or last_ema.pt (no optimizer state, but weights work)
    for name in ('best.pt', 'last_ema.pt'):
        cand = WEIGHTS_DRIVE / name
        if cand.is_file():
            resume = f' --resume {cand}'
            print('Resuming from', cand, '(weights only, no optimizer state)')
            break

cmd = f'''python -m src.train \\
  --data_root data \\
  --preset {PRESET} \\
  --batch_size {BATCH} \\
  --iters {ITERS} \\
  --gt_crop {GT_CROP} \\
  --num_workers 2 \\
  --val_every {VAL_EVERY} \\
  --save_every {SAVE_EVERY} \\
  --log_every {LOG_EVERY} \\
  --out_dir weights{resume}{NO_LPIPS}
'''
print(cmd)
!{cmd}

In [ ]:
# Mirror checkpoints to Drive (run after training / periodically)
import shutil
from pathlib import Path

src = Path('weights')
dst = Path('/content/drive/MyDrive/SpectraRestore/weights')
dst.mkdir(parents=True, exist_ok=True)
for f in src.glob('*.pt'):
    shutil.copy2(f, dst / f.name)
    print('copied', f.name)
!ls -lh /content/drive/MyDrive/SpectraRestore/weights | head

## 7 · Evaluate (KLA script)

Points `evaluate.py` at a folder of degraded images and writes restored outputs using the **same filenames** as the inputs.


In [ ]:
from pathlib import Path
import shutil

# Prefer Drive weights if local missing
for name in ('best.pt', 'last_ema.pt'):
    drive_w = Path('/content/drive/MyDrive/SpectraRestore/weights') / name
    local_w = Path('weights') / name
    if drive_w.is_file() and not local_w.is_file():
        Path('weights').mkdir(exist_ok=True)
        shutil.copy2(drive_w, local_w)
        print('restored', name, 'from Drive')

# Change INPUT to your test degraded folder
INPUT = 'data/val/degraded'          # or a KLA-released test folder
OUTPUT = 'outputs/val_restored'

# Outputs keep the SAME filenames as inputs (KLA scoring safety).
WEIGHTS = Path('weights/best.pt') if Path('weights/best.pt').is_file() else Path('weights/last_ema.pt')
assert WEIGHTS.is_file(), 'No checkpoint found. Run the training cell first.'
!python evaluate.py --input_dir {INPUT} --output_dir {OUTPUT} --weights {WEIGHTS}

# Optional: copy results to Drive
drive_out = Path('/content/drive/MyDrive/SpectraRestore/outputs')
drive_out.mkdir(parents=True, exist_ok=True)
!cp -r {OUTPUT} {drive_out}/
print('Saved to', drive_out)


## 8 · Preview a few restorations

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

out_dir = Path('outputs/val_restored')
deg_dir = Path('data/val/degraded')
gt_dir  = Path('data/val/gt')

# evaluate.py writes the SAME filenames as inputs
restored = sorted([p for p in out_dir.rglob('*') if p.is_file()])
assert restored, f'No outputs in {out_dir} — run the evaluate cell first'
sample = random.choice(restored)
stem = sample.stem

def load(p):
    if p.suffix == '.npy':
        a = np.load(p)
    else:
        a = np.asarray(Image.open(p)).astype(np.float32)
        if a.max() > 1.5:
            a = a / 255.0
    if a.ndim == 3:
        a = a[..., 0]
    return np.clip(a, 0, 1)

deg_path = next(deg_dir.glob(stem + '.*'), None)
gt_path = next(gt_dir.glob(stem + '.*'), None) if gt_dir.exists() else None

fig, axs = plt.subplots(1, 3 if gt_path else 2, figsize=(12, 4))
axs[0].imshow(load(deg_path), cmap='gray'); axs[0].set_title('Degraded'); axs[0].axis('off')
axs[1].imshow(load(sample), cmap='gray'); axs[1].set_title('Restored'); axs[1].axis('off')
if gt_path:
    axs[2].imshow(load(gt_path), cmap='gray'); axs[2].set_title('GT'); axs[2].axis('off')
plt.tight_layout(); plt.show()
print(sample)


## Colab disconnect survival

1. Always run the **copy weights to Drive** cell after training chunks  
2. Re-open this notebook → Mount Drive → Run All (auto-clones & auto-resumes)  
3. Keep the browser tab awake (or use Colab Pro) for long runs

Full design notes: see `SOLUTION.md` in the project.